In [1]:
import os
from beir import util
from beir.datasets.data_loader import GenericDataLoader
from sentence_transformers import SentenceTransformer
import torch
import numpy as np

c:\Users\Chaitanya's Laptop\Desktop\Data_Science\.venv\lib\site-packages\beir\util.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [2]:
# 1. Choose a manageable dataset for initial testing
dataset_name = "scifact" 
url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{dataset_name}.zip"

# 2. Define where to save it, then download and unzip
out_dir = os.path.join(os.getcwd(), "datasets")
data_path = util.download_and_unzip(url, out_dir)

# 3. Load the parsed data
# The 'test' split is standard for zero-shot evaluation in BEIR
corpus, queries, qrels = GenericDataLoader(data_folder=data_path).load(split="test")

# 4. Quick sanity check
print(f"Loaded {len(corpus)} documents.")
print(f"Loaded {len(queries)} queries.")
print(f"Loaded {len(qrels)} relevance judgments.")

# 5. Inspect the structure
sample_doc_id = list(corpus.keys())[0]
sample_query_id = list(queries.keys())[0]

print("\n--- Example Document ---")
print(f"ID: {sample_doc_id}")
print(f"Content: {corpus[sample_doc_id]}")

print("\n--- Example Query ---")
print(f"ID: {sample_query_id}")
print(f"Content: {queries[sample_query_id]}")

  0%|          | 0/5183 [00:00<?, ?it/s]

Loaded 5183 documents.
Loaded 300 queries.
Loaded 300 relevance judgments.

--- Example Document ---
ID: 4983
Content: {'text': 'Alterations of the architecture of cerebral white matter in the developing human brain can affect cortical development and result in functional disabilities. A line scan diffusion-weighted magnetic resonance imaging (MRI) sequence with diffusion tensor analysis was applied to measure the apparent diffusion coefficient, to calculate relative anisotropy, and to delineate three-dimensional fiber architecture in cerebral white matter in preterm (n = 17) and full-term infants (n = 7). To assess effects of prematurity on cerebral white matter development, early gestation preterm infants (n = 10) were studied a second time at term. In the central white matter the mean apparent diffusion coefficient at 28 wk was high, 1.8 microm2/ms, and decreased toward term to 1.2 microm2/ms. In the posterior limb of the internal capsule, the mean apparent diffusion coefficients at

In [3]:
# Assuming 'corpus' is already loaded from the BEIR SciFact dataset
document_ids = []
documents_text = []

for doc_id, doc_data in corpus.items():
    # Combine title and text for maximum context
    full_text = doc_data.get("title", "") + " " + doc_data.get("text", "")
    document_ids.append(doc_id)
    documents_text.append(full_text)

In [4]:
print(len(document_ids), len(documents_text))

5183 5183


In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"
bert_model = SentenceTransformer('all-MiniLM-L6-v2', device=device)

# 2. Encode the documents
# batch_size=32 is a safe default for a 8GB VRAM mobile GPU
print("Generating BERT embeddings...")
bert_embeddings = bert_model.encode(documents_text, batch_size=32, show_progress_bar=True)

# 3. Save the embeddings and the corresponding IDs to disk
np.save("bert_embeddings.npy", bert_embeddings)
# Saving doc_ids so the index row matches the actual document ID
with open("doc_ids.txt", "w", encoding="utf-8") as f:
    for doc_id in document_ids:
        f.write(f"{doc_id}\n")

print(f"Saved BERT embeddings with shape: {bert_embeddings.shape}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Generating BERT embeddings...


Batches:   0%|          | 0/162 [00:00<?, ?it/s]

Saved BERT embeddings with shape: (5183, 384)


In [7]:
import faiss
import numpy as np

# 1. Load your generated embeddings
embeddings = np.load("bert_embeddings.npy").astype('float32')

# 2. Initialize the index
# IndexFlatIP uses Inner Product (Cosine Similarity if vectors are normalized)
# For all-MiniLM-L6-v2, the dimension is 384
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)

# 3. Normalize vectors for Cosine Similarity
faiss.normalize_L2(embeddings)

# 4. Add documents to the index
index.add(embeddings)

print(f"Index built with {index.ntotal} documents.")

Index built with 5183 documents.


In [8]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.version.cuda)
print(torch.cuda.device_count())

2.5.1+cu121
True
12.1
1


In [9]:
def search(query_text, k=5):
    # 1. Convert query to vector
    query_vector = bert_model.encode([query_text]).astype('float32')
    
    # 2. Normalize query vector
    faiss.normalize_L2(query_vector)
    
    # 3. Search the index
    # 'D' is the similarity scores, 'I' is the indices (row numbers)
    distances, indices = index.search(query_vector, k)
    
    # 4. Map indices back to your original Doc IDs
    results = []
    for i in range(len(indices[0])):
        idx = indices[0][i]
        results.append({
            "doc_id": document_ids[idx],
            "score": float(distances[0][i]),
            "text": documents_text[idx]
        })
    return results

# Test it!
sample_query = list(queries.values())[0] # Take the first query from BEIR
print(f"Testing Query: {sample_query}")
hits = search(sample_query, k=3)

for hit in hits:
    print(f"\nScore: {hit['score']:.4f} | ID: {hit['doc_id']}")
    print(f"Snippet: {hit['text'][:200]}...")

Testing Query: 0-dimensional biomaterials show inductive properties.

Score: 0.3540 | ID: 29638116
Snippet: Complex Tissue and Disease Modeling using hiPSCs. Defined genetic models based on human pluripotent stem cells have opened new avenues for understanding disease mechanisms and drug screening. Many of ...

Score: 0.3306 | ID: 4346436
Snippet: Nonlinear Elasticity in Biological Gels Unlike most synthetic materials, biological materials often stiffen as they are deformed. This nonlinear elastic response, critical for the physiological functi...

Score: 0.2922 | ID: 3874000
Snippet: Tissue Mechanics Orchestrate Wnt-Dependent Human Embryonic Stem Cell Differentiation. Regenerative medicine is predicated on understanding the mechanisms regulating development and applying these cond...


In [10]:
def evaluate_search_engine(queries_dict, qrels_dict, k=10):
    total_precision = 0.0
    total_recall = 0.0
    valid_queries = 0
    
    print(f"Starting evaluation across {len(queries_dict)} queries...")
    
    # Loop through every query in the dataset
    for query_id, query_text in queries_dict.items():
        # Only evaluate queries that have ground-truth relevance judgments
        if query_id not in qrels_dict:
            continue
            
        # Get the ground truth: the set of relevant document IDs for this query
        # BEIR qrels structure: {query_id: {doc_id: score, doc_id: score}}
        relevant_docs = set(qrels_dict[query_id].keys())
        
        if len(relevant_docs) == 0:
            continue
            
        valid_queries += 1
        
        # 1. Retrieve the top-k documents using your existing search function
        results = search(query_text, k=k)
        retrieved_docs = [hit["doc_id"] for hit in results]
        
        # 2. Count the hits (intersection of retrieved and relevant)
        hits = sum(1 for doc_id in retrieved_docs if doc_id in relevant_docs)
        
        # 3. Calculate metrics for this specific query
        precision_at_k = hits / k
        recall_at_k = hits / len(relevant_docs)
        
        total_precision += precision_at_k
        total_recall += recall_at_k

    # Calculate the mean across all queries
    mean_precision = total_precision / valid_queries
    mean_recall = total_recall / valid_queries
    
    print("-" * 30)
    print(f"Evaluation Complete!")
    print(f"Total Valid Queries Evaluated: {valid_queries}")
    print(f"Mean Precision@{k}: {mean_precision:.4f}")
    print(f"Mean Recall@{k}: {mean_recall:.4f}")
    
    return mean_precision, mean_recall

# Run the evaluation for Top-10 documents
mean_p_10, mean_r_10 = evaluate_search_engine(queries, qrels, k=10)

Starting evaluation across 300 queries...
------------------------------
Evaluation Complete!
Total Valid Queries Evaluated: 300
Mean Precision@10: 0.0883
Mean Recall@10: 0.7833


In [11]:
from sentence_transformers.cross_encoder import CrossEncoder
import torch

# Load the Cross-Encoder and push it to the GPU 
# This handles the tokenization and prediction natively
device = "cuda" if torch.cuda.is_available() else "cpu"
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', device=device)
print("Cross-Encoder loaded successfully.")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

c:\Users\Chaitanya's Laptop\Desktop\Data_Science\.venv\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Chaitanya's Laptop\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Cross-Encoder loaded successfully.


In [12]:
def search_and_rerank(query_text, initial_k=100, final_top_n=10):
    """
    Two-stage retrieval pipeline.
    initial_k: How many documents to fetch with the fast Bi-Encoder.
    final_top_n: How many documents to return after Re-ranking.
    """
    # Stage 1: First-Stage Retrieval
    # Cast a wide net to maximize Recall
    initial_hits = search(query_text, k=initial_k)
    
    if not initial_hits:
        return []

    # Stage 2: Cross-Encoder Preparation
    # Format the inputs as a list of lists: [[query, doc_text], [query, doc_text], ...]
    cross_inp = [[query_text, hit["text"]] for hit in initial_hits]
    
    # Stage 3: Inference
    # Calculate the deep semantic similarity scores
    cross_scores = cross_encoder.predict(cross_inp)
    
    # Stage 4: Re-rank
    # Attach the new scores to the original hits
    for idx in range(len(cross_scores)):
        initial_hits[idx]["cross_score"] = float(cross_scores[idx])
        
    # Sort the hits descending based entirely on the Cross-Encoder's judgment
    reranked_hits = sorted(initial_hits, key=lambda x: x["cross_score"], reverse=True)
    
    # Return the highly refined top N documents
    return reranked_hits[:final_top_n]

In [13]:
# Test a single query
sample_query = list(queries.values())[0]
print(f"Testing Query: {sample_query}\n")

reranked_results = search_and_rerank(sample_query, initial_k=100, final_top_n=3)

for hit in reranked_results:
    print(f"Bi-Encoder Score: {hit['score']:.4f} | Cross-Encoder Logit: {hit['cross_score']:.4f}")
    print(f"Doc ID: {hit['doc_id']}")
    print(f"Snippet: {hit['text'][:150]}...\n")

Testing Query: 0-dimensional biomaterials show inductive properties.

Bi-Encoder Score: 0.2052 | Cross-Encoder Logit: -5.4163
Doc ID: 43385013
Snippet: Epithelial and mesenchymal subpopulations within normal basal breast cell lines exhibit distinct stem cell/progenitor properties. It has been proposed...

Bi-Encoder Score: 0.1931 | Cross-Encoder Logit: -7.4399
Doc ID: 10982689
Snippet: Nanotoxicology: An Emerging Discipline Evolving from Studies of Ultrafine Particles Although humans have been exposed to airborne nanosized particles ...

Bi-Encoder Score: 0.2095 | Cross-Encoder Logit: -7.4692
Doc ID: 6863070
Snippet: Three-dimensional superresolution colocalization of intracellular protein superstructures and the cell surface in live Caulobacter crescentus. Recentl...



In [15]:
def evaluate_reranker(queries_dict, qrels_dict, initial_k=100, final_k=10):
    total_precision = 0.0
    total_recall = 0.0
    valid_queries = 0
    
    print(f"Starting Two-Stage Evaluation across {len(queries_dict)} queries...")
    
    for query_id, query_text in queries_dict.items():
        if query_id not in qrels_dict:
            continue
            
        relevant_docs = set(qrels_dict[query_id].keys())
        if len(relevant_docs) == 0:
            continue
            
        valid_queries += 1
        
        # Call the new re-ranking pipeline instead of just search()
        results = search_and_rerank(query_text, initial_k=initial_k, final_top_n=final_k)
        retrieved_docs = [hit["doc_id"] for hit in results]
        
        hits = sum(1 for doc_id in retrieved_docs if doc_id in relevant_docs)
        
        total_precision += hits / final_k
        total_recall += hits / len(relevant_docs)

    print("-" * 30)
    print(f"Two-Stage Evaluation Complete!")
    print(f"Mean Precision@{final_k}: {total_precision / valid_queries:.4f}")
    print(f"Mean Recall@{final_k}: {total_recall / valid_queries:.4f}")

# Run it to see your new metrics
evaluate_reranker(queries, qrels, initial_k=100, final_k=10)

Starting Two-Stage Evaluation across 300 queries...
------------------------------
Two-Stage Evaluation Complete!
Mean Precision@10: 0.0917
Mean Recall@10: 0.8139


### **Word2Vev**

In [19]:
import gensim
from gensim.models import Word2Vec
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import string
import numpy as np

# Download necessary NLTK datasets (only runs once)
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('english'))

print("Cleaning text for Word2Vec...")
tokenized_docs = []
for text in documents_text:
    # Lowercase and remove punctuation
    text = text.lower().translate(str.maketrans('', '', string.punctuation))
    tokens = word_tokenize(text)
    # Remove stopwords
    clean_tokens = [word for word in tokens if word not in stop_words]
    tokenized_docs.append(clean_tokens)

# Train the Word2Vec model on the cleaned corpus
print("Training Word2Vec model...")
w2v_model = Word2Vec(sentences=tokenized_docs, vector_size=300, window=5, min_count=2, workers=4)

# Function to average word vectors into a document vector
def get_document_vector(model, doc_tokens):
    valid_words = [word for word in doc_tokens if word in model.wv.key_to_index]
    if not valid_words:
        return np.zeros(model.vector_size)
    return np.mean(model.wv[valid_words], axis=0)

print("Generating Word2Vec document embeddings...")
w2v_embeddings = np.array([get_document_vector(w2v_model, tokens) for tokens in tokenized_docs])

# Save the Word2Vec embeddings to disk
np.save("w2v_embeddings.npy", w2v_embeddings)

print(f"Successfully saved w2v_embeddings.npy with shape: {w2v_embeddings.shape}")

Cleaning text for Word2Vec...
Training Word2Vec model...
Generating Word2Vec document embeddings...
Successfully saved w2v_embeddings.npy with shape: (5183, 300)


In [20]:
import faiss
import numpy as np

# 1. Load the saved Word2Vec embeddings
w2v_embeddings = np.load("w2v_embeddings.npy").astype('float32')

# 2. Initialize the index for 300 dimensions
dimension_w2v = w2v_embeddings.shape[1] 
index_w2v = faiss.IndexFlatIP(dimension_w2v)

# 3. Normalize for Cosine Similarity and add to the index
faiss.normalize_L2(w2v_embeddings)
index_w2v.add(w2v_embeddings)

print(f"Word2Vec Index built with {index_w2v.ntotal} documents.")

Word2Vec Index built with 5183 documents.


In [21]:
import string
from nltk.tokenize import word_tokenize

def search_w2v(query_text, k=10):
    # 1. Clean the query text
    text = query_text.lower().translate(str.maketrans('', '', string.punctuation))
    tokens = word_tokenize(text)
    clean_tokens = [word for word in tokens if word not in stop_words]
    
    # 2. Vectorize the query (reusing your previously defined function)
    query_vector = get_document_vector(w2v_model, clean_tokens).astype('float32')
    
    # Reshape for FAISS: Needs to be a 2D array (1, dimension)
    query_vector = np.expand_dims(query_vector, axis=0)
    faiss.normalize_L2(query_vector)
    
    # 3. Search the Word2Vec index
    distances, indices = index_w2v.search(query_vector, k)
    
    # 4. Map indices back to original Doc IDs
    results = []
    for i in range(len(indices[0])):
        idx = indices[0][i]
        results.append({
            "doc_id": document_ids[idx],
            "score": float(distances[0][i]),
            "text": documents_text[idx]
        })
    return results

In [22]:
def evaluate_w2v_engine(queries_dict, qrels_dict, k=10):
    total_precision = 0.0
    total_recall = 0.0
    valid_queries = 0
    
    print(f"Starting Word2Vec evaluation across {len(queries_dict)} queries...")
    
    for query_id, query_text in queries_dict.items():
        if query_id not in qrels_dict:
            continue
            
        relevant_docs = set(qrels_dict[query_id].keys())
        if len(relevant_docs) == 0:
            continue
            
        valid_queries += 1
        
        # Call the Word2Vec search function
        results = search_w2v(query_text, k=k)
        retrieved_docs = [hit["doc_id"] for hit in results]
        
        hits = sum(1 for doc_id in retrieved_docs if doc_id in relevant_docs)
        
        total_precision += hits / k
        total_recall += hits / len(relevant_docs)

    mean_precision = total_precision / valid_queries
    mean_recall = total_recall / valid_queries
    
    print("-" * 30)
    print("Word2Vec Evaluation Complete!")
    print(f"Mean Precision@{k}: {mean_precision:.4f}")
    print(f"Mean Recall@{k}: {mean_recall:.4f}")

# Run it to see the classical NLP metrics
evaluate_w2v_engine(queries, qrels, k=10)

Starting Word2Vec evaluation across 300 queries...
------------------------------
Word2Vec Evaluation Complete!
Mean Precision@10: 0.0057
Mean Recall@10: 0.0500


In [23]:
import pickle

# Save the lists to a single file
with open("corpus_data.pkl", "wb") as f:
    pickle.dump({"ids": document_ids, "text": documents_text}, f)

print("Saved corpus_data.pkl successfully!")

Saved corpus_data.pkl successfully!
